# 03 — Time-series diagnostics and model eligibility

This notebook is the first formal econometric diagnostics stage. Run notebooks 01 and 02 first. The notebook reads the cleaned monthly data and the original quarterly GDP data; it does not use interpolated GDP.

The aim is **not** to select the model that gives the smallest p-values. It is to document the time-series properties of the candidate variables and establish which model classes are eligible under the research protocol.

Main steps:
1. Audit sample continuity and large changes before testing.
2. Test levels and first differences using ADF, Phillips–Perron (PP) and KPSS.
3. Run a one-break unit-root sensitivity test and record the estimated break date.
4. Keep quarterly GDP at quarterly frequency.
5. Screen model eligibility using reviewed integration-order evidence.
6. Provide gated parameter-stability, residual-diagnostic/HAC, Johansen cointegration and ARDL/UECM bounds workflows.
7. Keep candidate VAR lag selection, impulse-response uncertainty and chronological backtesting gated until stationarity is established.

The deterministic terms are tested as alternative specifications. A constant and a constant-plus-trend are not interchangeable; the final choice must be justified by the series plot and economic meaning.

In [ ]:
from pathlib import Path
import sys
import platform
import arch
import statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.api import VAR

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

# Resolve the project root whether Jupyter starts at the repository root or in notebooks/.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (path for path in candidates if (path / "data" / "processed" / "ghana_banking_monthly_clean.csv").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find data/processed/ghana_banking_monthly_clean.csv. "
        "Run notebook 01_data_cleaning.ipynb first."
    )

sys.path.insert(0, str(ROOT / "src"))
from ghana_banking.econometrics import (
    run_unit_root_suite,
    run_zivot_andrews_test,
    run_ols_stability_diagnostics,
    run_ols_residual_diagnostics,
    assess_model_eligibility,
    run_johansen_cointegration_test,
    fit_ardl_bounds_model,
    fit_vecm_model,
)

PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "figures"
PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print(f"Repository: {ROOT}")
environment = pd.DataFrame([{
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "statsmodels": statsmodels.__version__,
    "arch": arch.__version__,
}])
display(environment)
from ghana_banking.modeling import (
    VARModel,
    expanding_window_var_backtest,
    expanding_window_ar_backtest,
    forecast_ardl_scenarios,
    calibrate_historical_scenario_paths,
    calibrate_joint_historical_scenario_paths,
    run_ardl_scenario_sensitivity,
)


## 1. Load and audit the monthly sample

The primary monthly analysis excludes GDP. Confirm that the date index is monthly, unique and ordered. Any missing calendar month is reported; it is not silently inserted or imputed.

In [ ]:
monthly_path = PROCESSED / "ghana_banking_monthly_clean.csv"
monthly = pd.read_csv(monthly_path, parse_dates=["Date"], index_col="Date")
monthly = monthly.sort_index().apply(pd.to_numeric, errors="coerce")

if monthly.empty:
    raise ValueError("The cleaned monthly dataset is empty.")
if monthly.index.has_duplicates:
    raise ValueError("Duplicate monthly dates found.")
if not monthly.index.is_monotonic_increasing:
    raise ValueError("Monthly dates are not in chronological order.")

required_columns = [
    "Monetary Policy Rate (%)",
    "Consumer Price Index, All Items",
    "USD Exchange Rate, monthly averages",
    "Total Liquidity (M2+)",
    "Non Performing Loan Ratio",
]
missing_columns = sorted(set(required_columns) - set(monthly.columns))
if missing_columns:
    raise ValueError(f"Missing required monthly columns: {missing_columns}")

expected_months = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")
missing_months = expected_months.difference(monthly.index)

sample_audit = pd.DataFrame(
    {
        "first_valid": monthly[required_columns].apply(lambda col: col.first_valid_index()),
        "last_valid": monthly[required_columns].apply(lambda col: col.last_valid_index()),
        "observed": monthly[required_columns].notna().sum(),
        "missing": monthly[required_columns].isna().sum(),
        "missing_pct": 100 * monthly[required_columns].isna().mean(),
    }
)
display(sample_audit)

print(f"Monthly source coverage: {monthly.index.min():%Y-%m} to {monthly.index.max():%Y-%m}")
print(f"Rows in cleaned file: {len(monthly):,}")
print(f"Missing calendar months: {len(missing_months)}")
if len(missing_months):
    display(pd.DataFrame({"missing_month": missing_months}))

## 2. Construct the core monthly series

Positive price and quantity indices are analysed in logarithms. The NPL ratio and policy rate remain in their published percentage units. First differences of the NPL ratio and policy rate are therefore **percentage-point changes**; first differences of logged CPI, the exchange rate and M2+ are approximate monthly percentage changes.

M2+ receives a separate data-quality check because unusually large historical movements are visible in the source file. Such observations are flagged for source verification, not automatically deleted or winsorised.

In [ ]:
positive_columns = [
    "Consumer Price Index, All Items",
    "USD Exchange Rate, monthly averages",
    "Total Liquidity (M2+)",
]
for column in positive_columns:
    observed = monthly[column].dropna()
    if observed.empty or (observed <= 0).any():
        raise ValueError(f"{column} must contain positive observed values for log transformations.")

core_levels = pd.DataFrame(index=monthly.index)
core_levels["NPL ratio (%)"] = monthly["Non Performing Loan Ratio"]
core_levels["Policy rate (%)"] = monthly["Monetary Policy Rate (%)"]
core_levels["log CPI"] = np.log(monthly["Consumer Price Index, All Items"])
core_levels["log GHS per USD"] = np.log(monthly["USD Exchange Rate, monthly averages"])
core_levels["log M2+"] = np.log(monthly["Total Liquidity (M2+)"])

# Keep a named transformation table for interpretation and later modelling.
candidate_changes = pd.DataFrame(index=monthly.index)
candidate_changes["NPL change (percentage points)"] = core_levels["NPL ratio (%)"].diff()
candidate_changes["Policy-rate change (percentage points)"] = core_levels["Policy rate (%)"].diff()
candidate_changes["Inflation (monthly log change, %)"] = 100 * core_levels["log CPI"].diff()
candidate_changes["Cedi depreciation (monthly log change, %)"] = 100 * core_levels["log GHS per USD"].diff()
candidate_changes["M2+ growth (monthly log change, %)"] = 100 * core_levels["log M2+"].diff()

display(candidate_changes.describe().T)

## 3. Flag unusually large changes for source review

The robust z-score uses the median and median absolute deviation (MAD). A threshold of five is a screening rule only. The result is a review list, not an instruction to remove data. Check flagged dates against the original Bank of Ghana or GSS release and the relevant series definition before deciding how to treat them.

In [ ]:
def robust_change_flags(changes, threshold=5.0):
    """Return large-change flags using a median/MAD robust z-score."""
    rows = []
    for column in changes.columns:
        values = changes[column].replace([np.inf, -np.inf], np.nan).dropna()
        if values.empty:
            continue
        median = values.median()
        mad = (values - median).abs().median()
        if not np.isfinite(mad) or mad == 0:
            continue
        robust_z = 0.67448975 * (values - median) / mad
        flagged = robust_z.loc[robust_z.abs() >= threshold]
        for date, score in flagged.items():
            rows.append(
                {
                    "series": column,
                    "date": date,
                    "observed_change": values.loc[date],
                    "robust_z_score": score,
                }
            )
    return pd.DataFrame(rows, columns=["series", "date", "observed_change", "robust_z_score"])

change_flags = robust_change_flags(candidate_changes, threshold=5.0)
display(change_flags)

change_flags.to_csv(PROCESSED / "monthly_large_change_flags.csv", index=False)
print("Saved screening output: data/processed/monthly_large_change_flags.csv")
print("No observations have been removed or modified.")

## 4. ADF, PP and KPSS tests

The ADF and PP null hypotheses are that the series contains a unit root. The KPSS null is stationarity around the specified deterministic component. Agreement across tests is more informative than any one p-value, but no automatic integration-order label is assigned.

For levels, both a constant-only and a constant-plus-linear-trend specification are reported. For first differences, the constant-only specification is reported. The trend specification must ultimately be selected using the plot and economic rationale. ADF lag selection uses BIC; PP uses its documented automatic Newey–West bandwidth rule; KPSS uses automatic bandwidth selection.

The p-values are unadjusted, and the tests have finite-sample limitations. Treat conflicting or borderline evidence as inconclusive rather than forcing an I(0)/I(1) classification.

In [ ]:
def run_unit_root_suite_for_notebook(series_name, level_series, frequency):
    """Run the package diagnostic engine on levels and first differences."""
    records = []
    transformations = [
        ("level", level_series),
        ("first difference", level_series.diff()),
    ]

    for transformation, raw_series in transformations:
        values = raw_series.dropna()
        if len(values) < 40 or values.nunique() < 2:
            raise ValueError(
                f"Insufficient usable variation for {series_name} ({transformation}): "
                f"{len(values)} observations."
            )

        trend_specs = ["c", "ct"] if transformation == "level" else ["c"]
        for trend in trend_specs:
            test_results = run_unit_root_suite(
                series=values,
                series_name=series_name,
                deterministic=trend,
                adf_method="bic",
                frequency=frequency,
                transformation=transformation,
            )
            for result in test_results:
                records.append(
                    {
                        "frequency": result.frequency,
                        "sample_start": result.sample_start,
                        "sample_end": result.sample_end,
                        "input_observations": result.observations,
                        "series": result.series_name,
                        "transformation": transformation,
                        "deterministic_terms": result.deterministic,
                        "test": "PP" if result.test == "Phillips-Perron" else result.test,
                        "null_hypothesis": (
                            "stationarity" if result.test == "KPSS" else "unit root"
                        ),
                        "statistic": result.statistic,
                        "p_value": result.pvalue,
                        "lags_or_bandwidth": result.selected_lags,
                        "effective_nobs": result.effective_observations,
                        "critical_value_5pct": result.critical_values.get("5%"),
                        "lag_method": result.lag_method,
                    }
                )
    return records


monthly_unit_root_rows = []
for series_name in core_levels.columns:
    monthly_unit_root_rows.extend(
        run_unit_root_suite_for_notebook(
            series_name=series_name,
            level_series=core_levels[series_name],
            frequency="monthly",
        )
    )

monthly_unit_root_results = pd.DataFrame(monthly_unit_root_rows)
display(
    monthly_unit_root_results.sort_values(
        ["series", "transformation", "deterministic_terms", "test"]
    ).reset_index(drop=True)
)

monthly_unit_root_results.to_csv(
    PROCESSED / "monthly_unit_root_tests.csv", index=False
)
print("Saved: data/processed/monthly_unit_root_tests.csv")

### How to read the test combination

| ADF / PP | KPSS | Initial interpretation |
|---|---|---|
| Reject unit-root null | Do not reject stationarity null | Evidence is broadly consistent with stationarity under the tested deterministic terms. |
| Do not reject unit-root null | Reject stationarity null | Evidence is broadly consistent with a unit root under the tested deterministic terms. |
| Both nulls rejected | Conflicting evidence; possible breaks, nonlinearities, deterministic-term mismatch or finite-sample issues. |
| Neither null rejected | Tests are inconclusive; persistence and low power may be relevant. |

This table is a reading aid, not a mechanical classifier. Inspect plots, deterministic terms, lag choices, sample dates and break sensitivity before assigning an integration order.

## 5. One-break unit-root sensitivity

The Zivot–Andrews test allows one unknown break in the deterministic component under the unit-root null. It is included as a sensitivity check, not a universal fix for structural change. The test does not replace multiple-break analysis, parameter-stability tests or an economic explanation of candidate break periods.

The baseline specification allows a break in both the intercept and linear trend (`ct`) and uses BIC lag selection with a maximum of 12 lags. The reusable project function records the estimated break date, selected lag, critical values and sample metadata. The date is an estimated statistical breakpoint, not automatically the date of a known economic event.

In [ ]:
za_rows = []
for series_name in core_levels.columns:
    values = core_levels[series_name].dropna()
    za_result = run_zivot_andrews_test(
        series=values,
        series_name=series_name,
        regression="ct",
        max_lags=12,
        autolag="BIC",
        trim=0.15,
        frequency="monthly",
        transformation="level",
    )
    za_rows.append(za_result.to_dict())

za_results = pd.DataFrame(za_rows).set_index("series_name")
display(za_results)
za_results.to_csv(PROCESSED / "monthly_zivot_andrews_tests.csv")
print("Saved: data/processed/monthly_zivot_andrews_tests.csv")

## 6. Quarterly real GDP: separate frequency, separate test

Quarterly GDP is loaded from the original cleaned quarterly file. It is not joined to the monthly sample or interpolated. The log level and quarterly log growth are tested separately. Because the quarterly sample is shorter, results should be interpreted with particular caution.

In [ ]:
gdp_path = PROCESSED / "ghana_real_gdp_quarterly_clean.csv"
gdp = pd.read_csv(gdp_path)
if not {"Quarter", "GDP_Real"}.issubset(gdp.columns):
    raise ValueError("Quarterly GDP file must contain Quarter and GDP_Real.")

gdp["Quarter"] = pd.PeriodIndex(gdp["Quarter"], freq="Q")
gdp["GDP_Real"] = pd.to_numeric(gdp["GDP_Real"], errors="coerce")
gdp = gdp.dropna(subset=["Quarter", "GDP_Real"]).sort_values("Quarter")
gdp = gdp.drop_duplicates("Quarter").set_index("Quarter")

if (gdp["GDP_Real"] <= 0).any():
    raise ValueError("Quarterly real GDP must be positive before taking logs.")

log_gdp = np.log(gdp["GDP_Real"])
quarterly_unit_root_results = pd.DataFrame(
    run_unit_root_suite_for_notebook(
        series_name="log real GDP (constant 2013 prices)",
        level_series=log_gdp,
        frequency="quarterly",
    )
)
display(quarterly_unit_root_results)

quarterly_unit_root_results.to_csv(
    PROCESSED / "quarterly_gdp_unit_root_tests.csv", index=False
)
print(f"Quarterly GDP sample: {gdp.index.min()} to {gdp.index.max()}")
print(f"Saved: data/processed/quarterly_gdp_unit_root_tests.csv")

## 7. Parameter stability and model-eligibility gates

Parameter-stability tests are meaningful only after a defensible stationary regression specification has been chosen. The optional example below uses changes in the NPL ratio, policy rate, inflation, and cedi depreciation. It is disabled by default: first inspect the integration results and select a specification whose dependent variable and regressors are stationary in the estimation scale. Do not run OLS stability tests on an unexamined levels regression and interpret the result as valid evidence.

CUSUM tests cumulative recursive residual stability; Hansen's test assesses parameter constancy. Rolling-window coefficients are descriptive sensitivity evidence, not a formal break-date estimator. The results are conditional on the supplied regression and do not establish causal effects.

In [ ]:
RUN_PARAMETER_STABILITY = False

if RUN_PARAMETER_STABILITY:
    stability_data = pd.DataFrame(
        {
            "NPL_change_pp": candidate_changes["NPL change (percentage points)"],
            "Policy_rate_change_pp": candidate_changes["Policy-rate change (percentage points)"],
            "Inflation_mom_pct": candidate_changes["Inflation (monthly log change, %)"],
            "Cedi_depreciation_mom_pct": candidate_changes[
                "Cedi depreciation (monthly log change, %)"
            ],
        },
        index=monthly.index,
    ).dropna()

    stability_result = run_ols_stability_diagnostics(
        data=stability_data,
        dependent_variable="NPL_change_pp",
        regressors=[
            "Policy_rate_change_pp",
            "Inflation_mom_pct",
            "Cedi_depreciation_mom_pct",
        ],
    )
    display(pd.Series(stability_result.to_dict()).drop("rolling_coefficients"))
    rolling_coefficients = pd.DataFrame(stability_result.rolling_coefficients)
    display(rolling_coefficients.tail())
    rolling_coefficients.to_csv(
        PROCESSED / "rolling_ols_stability_coefficients.csv", index=False
    )
    print("Saved: data/processed/rolling_ols_stability_coefficients.csv")
else:
    print(
        "Parameter-stability regression not run. Enable only after confirming "
        "the selected equation is stationary in its estimation scale."
    )

### Model eligibility screening

After the unit-root evidence has been reviewed and working integration labels have been recorded, the helper below screens candidate model classes. It does not infer labels from p-values or decide cointegration. Replace the illustrative mapping with the documented assessment for the exact variables in a proposed equation.

In [ ]:
# Example only: replace these labels with the reviewed results for the
# exact variables included in the proposed model.
EXAMPLE_INTEGRATION_ASSESSMENTS = {
    "NPL ratio": "uncertain",
    "inflation": "uncertain",
    "cedi depreciation": "uncertain",
    "policy rate": "uncertain",
}

eligibility_review = assess_model_eligibility(EXAMPLE_INTEGRATION_ASSESSMENTS)
display(eligibility_review)

### Optional Johansen cointegration-rank test

Only use this after the included variables have been assessed as I(1), none is I(2), and a small system has been specified on economic grounds. This cell is disabled by default. The trace and maximum-eigenvalue rank tests can disagree; report both and examine lag/deterministic sensitivity. Cointegration rank is not evidence of causal effects.

In [ ]:
RUN_JOHANSEN_TEST = False

if RUN_JOHANSEN_TEST:
    # Explicitly select a small, economically justified I(1) system.
    JOHANSEN_VARIABLES = []
    if len(JOHANSEN_VARIABLES) < 2:
        raise ValueError("Specify at least two reviewed I(1) variables.")
    JOHANSEN_INTEGRATION_ASSESSMENTS = {}
    johansen_data = core_levels[JOHANSEN_VARIABLES].dropna()
    johansen_result = run_johansen_cointegration_test(
        data=johansen_data,
        integration_assessments=JOHANSEN_INTEGRATION_ASSESSMENTS,
        det_order=0,
        k_ar_diff=1,
        significance_level=0.05,
        frequency="monthly",
    )
    display(pd.Series(johansen_result.to_dict()))
    pd.DataFrame(
        {
            "rank_null": johansen_result.null_hypotheses,
            "trace_statistic": johansen_result.trace_statistics,
            "trace_critical_5pct": johansen_result.trace_critical_values,
            "max_eigen_statistic": johansen_result.max_eigen_statistics,
            "max_eigen_critical_5pct": johansen_result.max_eigen_critical_values,
        }
    ).to_csv(PROCESSED / "johansen_cointegration_rank.csv", index=False)
    print("Saved: data/processed/johansen_cointegration_rank.csv")
else:
    print(
        "Johansen testing not run. Confirm I(1) status and specify the "
        "cointegration system before enabling this cell."
    )

### Optional VECM estimation

Only enable this after all included variables have reviewed I(1) evidence and the Johansen trace and maximum-eigenvalue results have been examined. Supply the rank selected in the written decision record; do not choose it by looking for significant coefficients. The deterministic terms and lagged-difference order must also be justified. The VECM routine reports adjustment and cointegrating coefficients plus residual whiteness and normality diagnostics; it does not establish causal effects.



In [ ]:
RUN_VECM_ESTIMATION = False

if RUN_VECM_ESTIMATION:
    VECM_VARIABLES = []
    VECM_INTEGRATION_ASSESSMENTS = {}
    VECM_COINTEGRATION_RANK = None  # Enter only after reviewing Johansen rank evidence.

    if len(VECM_VARIABLES) < 2:
        raise ValueError("Specify at least two variables for the VECM.")
    if VECM_COINTEGRATION_RANK is None:
        raise ValueError("Enter the reviewed Johansen cointegration rank.")
    vecm_data = core_levels[VECM_VARIABLES].dropna()
    vecm_result = fit_vecm_model(
        data=vecm_data,
        integration_assessments=VECM_INTEGRATION_ASSESSMENTS,
        cointegration_rank=VECM_COINTEGRATION_RANK,
        k_ar_diff=1,
        deterministic="co",
        whiteness_lags=12,
    )
    vecm_record = vecm_result.to_dict()
    display(pd.Series({
        key: value for key, value in vecm_record.items()
        if key not in {
            "adjustment_coefficients", "cointegrating_vectors",
            "short_run_coefficients", "residual_covariance",
        }
    }))
    pd.DataFrame(
        vecm_record["adjustment_coefficients"], index=VECM_VARIABLES
    ).to_csv(PROCESSED / "vecm_adjustment_coefficients.csv")
    pd.DataFrame(
        vecm_record["cointegrating_vectors"], index=VECM_VARIABLES
    ).to_csv(PROCESSED / "vecm_cointegrating_vectors.csv")
    pd.DataFrame(
        vecm_record["short_run_coefficients"], index=VECM_VARIABLES
    ).to_csv(PROCESSED / "vecm_short_run_coefficients.csv")
    pd.DataFrame(
        vecm_record["residual_covariance"],
        index=VECM_VARIABLES, columns=VECM_VARIABLES,
    ).to_csv(PROCESSED / "vecm_residual_covariance.csv")
    print("Saved VECM parameters and residual covariance under data/processed/.")
else:
    print(
        "VECM not estimated. Confirm I(1) status, review Johansen rank, "
        "and specify the system before enabling this cell."
    )

### Optional ARDL/UECM bounds workflow

The primary monthly equation can use ARDL bounds testing only after the exact dependent variable and regressors have been reviewed as I(0)/I(1), with no possible I(2) variable. An all-I(0) system should normally proceed through stationary dynamic regression rather than unnecessary bounds testing. The model below is disabled until the variables, transformations, integration assessments and deterministic case have been entered explicitly.

The routine selects lag orders by the chosen information criterion, estimates the ARDL and its UECM representation, and reports lower/upper bounds p-values. Long-run multipliers are returned only when the upper-bound p-value rejects the no-level-relationship null. A lag-selection result that cannot be represented as a UECM is rejected rather than silently altered.

In [ ]:
RUN_ARDL_BOUNDS_TEST = False

if RUN_ARDL_BOUNDS_TEST:
    ARDL_DEPENDENT_VARIABLE = "NPL ratio (%)"
    ARDL_REGRESSORS = []
    ARDL_INTEGRATION_ASSESSMENTS = {}

    ardl_candidate_data = pd.concat([core_levels, candidate_changes], axis=1)
    if not ARDL_REGRESSORS:
        raise ValueError("Specify the economically justified ARDL regressors.")
    ardl_training_data = ardl_candidate_data[[ARDL_DEPENDENT_VARIABLE] + ARDL_REGRESSORS].dropna()
    if len(ardl_training_data) < 60:
        raise ValueError("Fewer than 60 complete observations remain for the ARDL model.")
    print(
        f"ARDL complete-case sample: {ardl_training_data.index.min()} to "
        f"{ardl_training_data.index.max()} ({len(ardl_training_data)} observations)."
    )
    ardl_result = fit_ardl_bounds_model(
        data=ardl_training_data,
        dependent_variable=ARDL_DEPENDENT_VARIABLE,
        regressors=ARDL_REGRESSORS,
        integration_assessments=ARDL_INTEGRATION_ASSESSMENTS,
        max_ar_lag=6,
        max_distributed_lag=6,
        selection_ic="bic",
        trend="c",
        bounds_case=3,
        alpha=0.05,
        asymptotic_bounds=True,
    )
    ardl_record = ardl_result.to_dict()
    display(pd.Series({
        key: value for key, value in ardl_record.items()
        if key not in {"ardl_parameters", "uecm_parameters", "long_run_multipliers"}
    }))
    display(pd.DataFrame(ardl_record["ardl_parameters"]))
    display(pd.DataFrame(ardl_record["uecm_parameters"]))
    pd.DataFrame([{
        "bounds_decision": ardl_record["bounds_decision"],
        "bounds_statistic": ardl_record["bounds_statistic"],
        "lower_pvalue": ardl_record["bounds_lower_pvalue"],
        "upper_pvalue": ardl_record["bounds_upper_pvalue"],
        "long_run_multipliers": str(ardl_record["long_run_multipliers"]),
    }]).to_csv(PROCESSED / "ardl_bounds_results.csv", index=False)
    print("Saved: data/processed/ardl_bounds_results.csv")
else:
    print(
        "ARDL bounds workflow not run. Complete the integration decision record, "
        "choose the model variables and justify the deterministic case first."
    )

The screening helper deliberately blocks standard ARDL bounds testing when any series may be I(2) or unresolved, and blocks model screening when integration evidence is uncertain. A mixed I(0)/I(1) set is only a candidate for ARDL bounds testing; bounds inference and diagnostics are still required. An all-I(1) set is only a candidate for VECM after a cointegration-rank test. An all-I(0) set can support stationary dynamic regression or VAR, subject to model diagnostics.

## 8. Candidate VAR lag-order exercise (deliberately gated)

Do not execute lag selection until you have reviewed the unit-root results and decided that the candidate transformed variables are appropriate for a stationary VAR. The flag is False by default.

If enabled, the example uses four variables and a maximum of six monthly lags. It reports information-criterion choices for review; it does not automatically designate the AIC, BIC or HQIC winner as the final model. The final choice must account for parameter count, sample size, residual diagnostics and economic rationale.

In [ ]:
RUN_CANDIDATE_VAR_LAG_SELECTION = False

if RUN_CANDIDATE_VAR_LAG_SELECTION:
    candidate_var = pd.DataFrame(
        {
            "NPL_change_pp": candidate_changes["NPL change (percentage points)"],
            "Policy_rate_change_pp": candidate_changes["Policy-rate change (percentage points)"],
            "Inflation_mom_pct": candidate_changes["Inflation (monthly log change, %)"],
            "Cedi_depreciation_mom_pct": candidate_changes[
                "Cedi depreciation (monthly log change, %)"
            ],
        },
        index=monthly.index,
    ).dropna()

    n_obs, n_variables = candidate_var.shape
    max_estimable_lags = (n_obs - n_variables - 1) // (n_variables + 1)
    max_lags_to_check = min(6, max_estimable_lags)

    if max_lags_to_check < 1:
        raise ValueError("The complete-case sample is too small for a VAR lag-order exercise.")

    lag_selection = VAR(candidate_var).select_order(maxlags=max_lags_to_check)
    lag_orders = pd.DataFrame([lag_selection.selected_orders])
    display(lag_orders)
    display(pd.DataFrame(lag_selection.ics))

    lag_orders.to_csv(PROCESSED / "candidate_var_selected_lags.csv", index=False)
    pd.DataFrame(lag_selection.ics).to_csv(
        PROCESSED / "candidate_var_information_criteria.csv"
    )
    print(
        f"Candidate VAR: {n_obs} observations, {n_variables} variables, "
        f"maximum lags checked = {max_lags_to_check}."
    )
else:
    print(
        "Lag selection not run. Review the unit-root and transformation evidence, "
        "then set RUN_CANDIDATE_VAR_LAG_SELECTION = True only if a stationary VAR is eligible."
    )

## 9. VAR impulse-response uncertainty and chronological forecast backtesting

The two workflows below are optional and disabled by default. Enable them only after the integration-order review supports a stationary transformed VAR, the variable set and lag order have been justified, and residual/stability diagnostics have been reviewed. The forecast exercise uses expanding windows and never trains on observations after the forecast origin. It compares the VAR with an expanding historical-mean benchmark.

Impulse-response bands are pointwise Monte Carlo simulation bands conditional on the fitted VAR. Reduced-form responses are not structural shocks. Cholesky-orthogonalized responses depend on variable ordering and require a defensible ordering argument. Neither the bands nor the VAR backtest identify causal effects.



In [ ]:
RUN_VAR_CHRONOLOGICAL_BACKTEST = False

if RUN_VAR_CHRONOLOGICAL_BACKTEST:
    stationary_var_data = pd.DataFrame(
        {
            "NPL_change_pp": candidate_changes["NPL change (percentage points)"],
            "Policy_rate_change_pp": candidate_changes["Policy-rate change (percentage points)"],
            "Inflation_mom_pct": candidate_changes["Inflation (monthly log change, %)"],
            "Cedi_depreciation_mom_pct": candidate_changes[
                "Cedi depreciation (monthly log change, %)"
            ],
        },
        index=monthly.index,
    ).dropna()

    backtest_result = expanding_window_var_backtest(
        data=stationary_var_data,
        lag_order=2,  # Fixed ex ante; repeat selection inside each origin if data-driven.
        initial_window=60,
        horizons=[1, 3, 6],
        step=1,
    )
    backtest_record = backtest_result.to_dict()
    display(pd.DataFrame(backtest_record["metrics"]))
    pd.DataFrame(backtest_record["metrics"]).to_csv(
        PROCESSED / "var_expanding_window_backtest_metrics.csv", index=False
    )
    pd.DataFrame(backtest_record["forecast_records"]).to_csv(
        PROCESSED / "var_expanding_window_backtest_forecasts.csv", index=False
    )
    print("Saved chronological VAR backtest metrics and forecasts.")
else:
    print(
        "VAR backtest not run. Confirm stationarity, model adequacy and lag "
        "selection before enabling the expanding-window evaluation."
    )

In [ ]:
RUN_VAR_IRF_MONTE_CARLO_BANDS = False

if RUN_VAR_IRF_MONTE_CARLO_BANDS:
    stationary_var_data = pd.DataFrame(
        {
            "NPL_change_pp": candidate_changes["NPL change (percentage points)"],
            "Policy_rate_change_pp": candidate_changes["Policy-rate change (percentage points)"],
            "Inflation_mom_pct": candidate_changes["Inflation (monthly log change, %)"],
            "Cedi_depreciation_mom_pct": candidate_changes[
                "Cedi depreciation (monthly log change, %)"
            ],
        },
        index=monthly.index,
    ).dropna()

    selected_var = VARModel(lags=2).fit(stationary_var_data, use_diff=False)
    selected_var_diagnostics = selected_var.get_diagnostics(whiteness_lags=12)
    if not selected_var_diagnostics["stable"]:
        raise ValueError("VAR is unstable; do not interpret impulse responses.")
    if selected_var_diagnostics["whiteness_pvalue"] < 0.05:
        raise ValueError("VAR residual whiteness is rejected; revise the specification first.")

    irf_bands = selected_var.impulse_response_bands(
        periods=12,
        orthogonalized=False,
        replications=1000,
        significance_level=0.05,
        seed=12345,
    )
    lower = np.asarray(irf_bands["lower"])
    upper = np.asarray(irf_bands["upper"])
    irf_rows = []
    for horizon in range(lower.shape[0]):
        for response_index, response in enumerate(irf_bands["variables"]):
            for impulse_index, impulse in enumerate(irf_bands["variables"]):
                irf_rows.append({
                    "horizon": horizon,
                    "response": response,
                    "impulse": impulse,
                    "lower_95": lower[horizon, response_index, impulse_index],
                    "upper_95": upper[horizon, response_index, impulse_index],
                })
    pd.DataFrame(irf_rows).to_csv(
        PROCESSED / "var_reduced_form_irf_monte_carlo_bands.csv", index=False
    )
    print("Saved pointwise reduced-form IRF simulation bands.")
else:
    print(
        "VAR IRF bands not run. Confirm VAR stability and residual whiteness "
        "before enabling the Monte Carlo simulation."
    )

## 10. Diagnostic plots and files

Inspect the level and transformed series before deciding whether a constant, trend or break specification is economically plausible. The vertical scales differ across panels.

Files written by this notebook:
- monthly_unit_root_tests.csv
- monthly_zivot_andrews_tests.csv (including estimated break dates and critical values)
- monthly_large_change_flags.csv
- quarterly_gdp_unit_root_tests.csv
- optional rolling_ols_stability_coefficients.csv
- optional johansen_cointegration_rank.csv
- optional ardl_bounds_results.csv
- optional candidate_var_selected_lags.csv and candidate_var_information_criteria.csv

In [ ]:
plot_series = {
    "NPL ratio (%)": core_levels["NPL ratio (%)"],
    "Policy rate (%)": core_levels["Policy rate (%)"],
    "log CPI": core_levels["log CPI"],
    "log GHS per USD": core_levels["log GHS per USD"],
    "log M2+": core_levels["log M2+"],
}
fig, axes = plt.subplots(len(plot_series), 1, figsize=(13, 13), sharex=True)
for axis, (label, values) in zip(axes, plot_series.items()):
    axis.plot(values.index, values, linewidth=1.3)
    axis.set_ylabel(label)
    axis.grid(alpha=0.25)
axes[-1].set_xlabel("Month")
fig.suptitle("Candidate monthly series for time-series modelling", y=1.002)
fig.tight_layout()
fig.savefig(FIGURES / "monthly_candidate_series_for_modelling.png", dpi=160, bbox_inches="tight")
plt.show()

print("Diagnostic outputs saved under data/processed/.")
print("No model has been estimated and no integration order has been assigned automatically.")

## 11. Residual specification tests and HAC inference

Only run these diagnostics after choosing a defensible stationary regression specification and its estimation scale. The example uses the changes constructed above and remains disabled until the integration review supports that equation. Breusch–Godfrey tests residual serial correlation; Breusch–Pagan and White test heteroskedasticity; ARCH-LM tests conditional variance dynamics; Jarque–Bera tests residual normality; RESET checks functional-form adequacy. Durbin–Watson is reported as a descriptive serial-correlation statistic. Newey–West/HAC standard errors are shown beside conventional OLS inference.

A rejection is a diagnostic signal, not an automatic repair. HAC covariance estimation does not fix non-stationarity, endogeneity, omitted variables or a misspecified conditional mean. Report the HAC lag truncation and examine sensitivity.



In [ ]:
RUN_RESIDUAL_DIAGNOSTICS = False

if RUN_RESIDUAL_DIAGNOSTICS:
    residual_model_data = pd.DataFrame(
        {
            "NPL_change_pp": candidate_changes["NPL change (percentage points)"],
            "Policy_rate_change_pp": candidate_changes["Policy-rate change (percentage points)"],
            "Inflation_mom_pct": candidate_changes["Inflation (monthly log change, %)"],
            "Cedi_depreciation_mom_pct": candidate_changes[
                "Cedi depreciation (monthly log change, %)"
            ],
        },
        index=monthly.index,
    ).dropna()

    residual_result = run_ols_residual_diagnostics(
        data=residual_model_data,
        dependent_variable="NPL_change_pp",
        regressors=[
            "Policy_rate_change_pp",
            "Inflation_mom_pct",
            "Cedi_depreciation_mom_pct",
        ],
        autocorrelation_lags=12,
        arch_lags=12,
    )
    residual_record = residual_result.to_dict()
    display(pd.DataFrame.from_dict(residual_record["residual_tests"], orient="index"))
    display(pd.DataFrame(residual_record["conventional_parameters"]))
    display(pd.DataFrame(residual_record["hac_parameters"]))
    pd.DataFrame.from_dict(
        residual_record["residual_tests"], orient="index"
    ).to_csv(PROCESSED / "ols_residual_diagnostics.csv")
    pd.DataFrame(residual_record["hac_parameters"]).to_csv(
        PROCESSED / "ols_hac_coefficients.csv", index=False
    )
    print("Saved residual diagnostics and HAC coefficients under data/processed/.")
else:
    print(
        "Residual diagnostics not run. Confirm that the selected regression "
        "is stationary in its estimation scale before enabling this cell."
    )

## 12. Decision record before model estimation

Complete this decision record after reviewing the exported tables and plots. Do not fill it with conclusions that the test evidence does not support.

| Variable | Deterministic terms justified? | ADF / PP evidence | KPSS evidence | Break sensitivity | Working integration assessment | Eligible model treatment |
|---|---|---|---|---|---|---|
| NPL ratio | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| Policy rate | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| log CPI | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| log GHS per USD | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| log M2+ | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |
| log real GDP (quarterly) | To be assessed | To be assessed | To be assessed | To be assessed | Pending | Pending |

### Model eligibility rules

- ARDL bounds testing is considered only where no included variable is I(2) and the deterministic case and sample support the test.
- VECM is considered where the included system variables are I(1) and cointegration rank evidence supports a cointegrating relationship.
- Stationary VAR is considered only after the chosen transformations produce a defensible stationary system.
- If evidence is conflicting, the order of integration is uncertain or structural breaks dominate the results, report that uncertainty and revisit the specification instead of forcing a model choice.

### References for test implementations

- ADF: Dickey and Fuller (1979), https://doi.org/10.1080/01621459.1979.10482531
- PP: Phillips and Perron (1988), https://doi.org/10.1093/biomet/75.2.335
- KPSS: Kwiatkowski, Phillips, Schmidt and Shin (1992), https://doi.org/10.1016/0304-4076(92)90104-Y
- Zivot–Andrews: Zivot and Andrews (1992), https://doi.org/10.1080/07350015.1992.10509904

The Phillips–Perron routine is provided by `arch`; the reusable Zivot–Andrews wrapper uses `statsmodels.tsa.stattools.zivot_andrews` so the estimated break index/date can be retained. See the statsmodels time-series documentation for implementation details.

## 13. Univariate AR benchmark, prediction intervals and no-change comparison

This optional workflow benchmarks the NPL-change series against a univariate autoregression, a no-change forecast and an expanding historical mean. It uses expanding chronological windows and pointwise prediction intervals. The input must be a defensible stationary modelling scale; the lag order is prespecified here and should be justified from training-sample diagnostics. This is a predictive benchmark, not a causal model.


In [ ]:
RUN_AR_BENCHMARK_BACKTEST = False

if RUN_AR_BENCHMARK_BACKTEST:
    ar_benchmark_data = candidate_changes[["NPL change (percentage points)"]].dropna()
    ar_benchmark_result = expanding_window_ar_backtest(
        data=ar_benchmark_data,
        lag_order=2,  # Prespecify and justify; do not tune on the evaluation sample.
        initial_window=60,
        horizons=[1, 3, 6],
        step=1,
        significance_level=0.05,
    ).to_dict()
    ar_metrics = pd.DataFrame(ar_benchmark_result["metrics"])
    display(ar_metrics)
    ar_metrics.to_csv(PROCESSED / "npl_ar_backtest_metrics.csv", index=False)
    pd.DataFrame(ar_benchmark_result["forecast_records"]).to_csv(
        PROCESSED / "npl_ar_backtest_forecasts.csv", index=False
    )
    print("Saved AR benchmark metrics and forecast-level records.")
else:
    print("AR benchmark backtest disabled. Review stationarity and the prespecified lag order first.")

## 14. Historical calibration of baseline, adverse and severe macro paths

This optional workflow uses the historical AR(1) innovation distribution for each selected ARDL regressor. The adverse direction is an explicit economic assumption and must be justified variable by variable. The calibration is marginal, not joint: it does not preserve cross-variable shock dependence and does not produce a joint tail probability. Do not label these paths as official supervisory scenarios. The ARDL bounds workflow must first support a level relationship.


In [ ]:
RUN_HISTORICAL_SCENARIO_CALIBRATION = False

if RUN_HISTORICAL_SCENARIO_CALIBRATION:
    if not RUN_ARDL_BOUNDS_TEST:
        raise ValueError("Run and review the ARDL bounds workflow before scenario calibration.")
    if ardl_result.bounds_decision != "evidence_of_level_relationship":
        raise ValueError("Dynamic level scenarios require supported ARDL bounds evidence.")

    # Explicitly review these economic directions before enabling the workflow.
    SCENARIO_ADVERSE_DIRECTIONS = {
        # Example only: "Inflation (monthly log change, %)": 1,
        # Example only: "Cedi depreciation (monthly log change, %)": 1,
    }
    SCENARIO_INTEGRATED_VARIABLES = [
        variable for variable, order in ARDL_INTEGRATION_ASSESSMENTS.items()
        if variable in ARDL_REGRESSORS and order == "I(1)"
    ]
    if set(SCENARIO_ADVERSE_DIRECTIONS) != set(ARDL_REGRESSORS):
        raise ValueError("Set an explicit +1/-1 adverse direction for every ARDL regressor.")

    scenario_history = ardl_training_data[ARDL_REGRESSORS]
    calibrated_paths = calibrate_historical_scenario_paths(
        historical_exog=scenario_history,
        horizon=12,
        adverse_directions=SCENARIO_ADVERSE_DIRECTIONS,
        integrated_variables=SCENARIO_INTEGRATED_VARIABLES,
        adverse_quantile=0.90,
        severe_quantile=0.975,
        persistence=0.70,
    )

    from statsmodels.tsa.ardl import ARDL
    scenario_ardl_fit = ARDL(
        endog=ardl_training_data[ARDL_DEPENDENT_VARIABLE],
        lags=ardl_result.selected_ar_lag,
        exog=ardl_training_data[ARDL_REGRESSORS],
        order=ardl_result.selected_distributed_lags,
        trend=ardl_result.deterministic_trend,
    ).fit()
    scenario_forecasts = forecast_ardl_scenarios(
        fitted_results=scenario_ardl_fit,
        scenarios=calibrated_paths,
        baseline_name="Baseline",
        significance_level=0.05,
    )
    display(scenario_forecasts)
    scenario_forecasts.to_csv(
        PROCESSED / "ardl_historical_scenario_forecasts.csv", index=False
    )
    pd.DataFrame(calibrated_paths["Adverse"].attrs["calibration"]).to_csv(
        PROCESSED / "ardl_scenario_calibration_metadata.csv", index=False
    )
    for scenario_name, scenario_path in calibrated_paths.items():
        scenario_path.to_csv(
            PROCESSED / f"ardl_macro_path_{scenario_name.lower()}.csv"
        )
    print("Saved calibrated macro paths and conditional ARDL scenario forecasts.")
else:
    print(
        "Historical scenario calibration disabled. Review ARDL bounds evidence, "
        "integration classifications and adverse-direction assumptions first."
    )

## 15. Sensitivity to scenario quantiles and shock persistence

This optional exercise holds the selected ARDL model and historical sample fixed while varying the marginal innovation quantiles and the assumed geometric persistence of macro shocks. It measures sensitivity to scenario assumptions, not parameter uncertainty or model-selection uncertainty.


In [ ]:
RUN_SCENARIO_SENSITIVITY = False

if RUN_SCENARIO_SENSITIVITY:
    if not RUN_ARDL_BOUNDS_TEST or ardl_result.bounds_decision != "evidence_of_level_relationship":
        raise ValueError("Review and support the ARDL bounds result before scenario sensitivity analysis.")
    from statsmodels.tsa.ardl import ARDL
    scenario_ardl_fit = ARDL(
        endog=ardl_training_data[ARDL_DEPENDENT_VARIABLE],
        lags=ardl_result.selected_ar_lag,
        exog=ardl_training_data[ARDL_REGRESSORS],
        order=ardl_result.selected_distributed_lags,
        trend=ardl_result.deterministic_trend,
    ).fit()
    SCENARIO_ADVERSE_DIRECTIONS = {
        # Explicitly specify +1/-1 for every selected ARDL regressor.
    }
    SCENARIO_INTEGRATED_VARIABLES = [
        variable for variable, order in ARDL_INTEGRATION_ASSESSMENTS.items()
        if variable in ARDL_REGRESSORS and order == "I(1)"
    ]
    if set(SCENARIO_ADVERSE_DIRECTIONS) != set(ARDL_REGRESSORS):
        raise ValueError("Set an explicit adverse direction for every ARDL regressor.")

    sensitivity_result = run_ardl_scenario_sensitivity(
        fitted_results=scenario_ardl_fit,
        historical_exog=ardl_training_data[ARDL_REGRESSORS],
        horizon=12,
        adverse_directions=SCENARIO_ADVERSE_DIRECTIONS,
        integrated_variables=SCENARIO_INTEGRATED_VARIABLES,
        quantile_pairs=[(0.90, 0.975), (0.95, 0.99)],
        persistence_values=[0.30, 0.70, 0.90],
        significance_level=0.05,
    ).to_dict()
    sensitivity_summary = pd.DataFrame(sensitivity_result["summary"])
    sensitivity_forecasts = pd.DataFrame(sensitivity_result["forecasts"])
    display(sensitivity_summary)
    sensitivity_summary.to_csv(
        PROCESSED / "ardl_scenario_sensitivity_summary.csv", index=False
    )
    sensitivity_forecasts.to_csv(
        PROCESSED / "ardl_scenario_sensitivity_forecasts.csv", index=False
    )
    print("Saved scenario-assumption sensitivity summaries and horizon-level forecasts.")
else:
    print("Scenario sensitivity disabled. Prespecify quantile and persistence grids before enabling.")

## 16. Joint historical macro scenario calibration

This optional method selects contiguous blocks of aligned AR(1) residual vectors using an equal-weight, direction-adjusted severity score. It preserves observed cross-variable alignment and within-block ordering. It is a historical-analogue scenario, not a joint probability model or structural stress test.


In [ ]:
RUN_JOINT_SCENARIO_CALIBRATION = False

if RUN_JOINT_SCENARIO_CALIBRATION:
    if not RUN_ARDL_BOUNDS_TEST or ardl_result.bounds_decision != "evidence_of_level_relationship":
        raise ValueError("Review and support the ARDL bounds result before joint scenario calibration.")
    from statsmodels.tsa.ardl import ARDL
    joint_scenario_ardl_fit = ARDL(
        endog=ardl_training_data[ARDL_DEPENDENT_VARIABLE],
        lags=ardl_result.selected_ar_lag,
        exog=ardl_training_data[ARDL_REGRESSORS],
        order=ardl_result.selected_distributed_lags,
        trend=ardl_result.deterministic_trend,
    ).fit()
    JOINT_SCENARIO_ADVERSE_DIRECTIONS = {
        # Explicitly specify +1/-1 for every selected ARDL regressor.
    }
    JOINT_SCENARIO_INTEGRATED_VARIABLES = [
        variable for variable, order in ARDL_INTEGRATION_ASSESSMENTS.items()
        if variable in ARDL_REGRESSORS and order == "I(1)"
    ]
    if set(JOINT_SCENARIO_ADVERSE_DIRECTIONS) != set(ARDL_REGRESSORS):
        raise ValueError("Set an explicit adverse direction for every ARDL regressor.")

    joint_paths = calibrate_joint_historical_scenario_paths(
        historical_exog=ardl_training_data[ARDL_REGRESSORS],
        horizon=12,
        adverse_directions=JOINT_SCENARIO_ADVERSE_DIRECTIONS,
        integrated_variables=JOINT_SCENARIO_INTEGRATED_VARIABLES,
        adverse_quantile=0.90,
        severe_quantile=0.975,
    )
    joint_forecasts = forecast_ardl_scenarios(
        fitted_results=joint_scenario_ardl_fit,
        scenarios=joint_paths,
        baseline_name="Baseline",
    )
    display(joint_forecasts)
    joint_forecasts.to_csv(PROCESSED / "ardl_joint_scenario_forecasts.csv", index=False)
    print("Saved joint historical residual-block scenario forecasts.")

    RUN_JOINT_SCENARIO_WEIGHT_SENSITIVITY = False
    if RUN_JOINT_SCENARIO_WEIGHT_SENSITIVITY:
        # Supply at least two prespecified, economically justified weight schemes.
        JOINT_SCENARIO_WEIGHT_SCHEMES = {
            # "equal": {variable: 1.0 for variable in ARDL_REGRESSORS},
            # "fx_emphasis": {"inflation": 1.0, "cedi_depreciation": 2.0, "policy_rate_change": 1.0},
        }
        if len(JOINT_SCENARIO_WEIGHT_SCHEMES) < 2:
            raise ValueError("Specify at least two named severity-weight schemes.")
        weight_sensitivity_records = []
        for scheme_name, scheme_weights in JOINT_SCENARIO_WEIGHT_SCHEMES.items():
            weighted_paths = calibrate_joint_historical_scenario_paths(
                historical_exog=ardl_training_data[ARDL_REGRESSORS],
                horizon=12,
                adverse_directions=JOINT_SCENARIO_ADVERSE_DIRECTIONS,
                integrated_variables=JOINT_SCENARIO_INTEGRATED_VARIABLES,
                adverse_quantile=0.90,
                severe_quantile=0.975,
                severity_weights=scheme_weights,
            )
            weighted_forecasts = forecast_ardl_scenarios(
                fitted_results=joint_scenario_ardl_fit,
                scenarios=weighted_paths,
                baseline_name="Baseline",
            )
            for scenario_name in ("Adverse", "Severe"):
                scenario_rows = weighted_forecasts.loc[
                    weighted_forecasts["scenario"] == scenario_name
                ].sort_values("period")
                metadata = weighted_paths[scenario_name].attrs["calibration"]
                weight_sensitivity_records.append({
                    "weight_scheme": scheme_name,
                    "scenario": scenario_name,
                    "terminal_forecast": float(scenario_rows["forecast"].iloc[-1]),
                    "terminal_change_vs_baseline": float(
                        scenario_rows["change_vs_baseline"].iloc[-1]
                    ),
                    "peak_forecast": float(scenario_rows["forecast"].max()),
                    "peak_period": str(scenario_rows.loc[scenario_rows["forecast"].idxmax(), "period"]),
                    "selected_block_start": metadata[
                        "severe_block_start_period" if scenario_name == "Severe"
                        else "adverse_block_start_period"
                    ],
                    "selected_block_end": metadata[
                        "severe_block_end_period" if scenario_name == "Severe"
                        else "adverse_block_end_period"
                    ],
                    "selected_block_score": metadata[
                        "severe_block_score" if scenario_name == "Severe"
                        else "adverse_block_score"
                    ],
                    "normalized_weights": metadata["severity_weights"],
                })
        joint_weight_sensitivity = pd.DataFrame(weight_sensitivity_records)
        display(joint_weight_sensitivity)
        joint_weight_sensitivity.to_csv(
            PROCESSED / "ardl_joint_scenario_weight_sensitivity.csv", index=False
        )
        print("Saved joint scenario weight-sensitivity results.")


    RUN_JOINT_SCENARIO_BLOCK_LENGTH_SENSITIVITY = False
    if RUN_JOINT_SCENARIO_BLOCK_LENGTH_SENSITIVITY:
        # Prespecify episode lengths; each must be >= the 12-period forecast horizon.
        JOINT_SCENARIO_BLOCK_LENGTHS = [12, 18, 24]
        if len(JOINT_SCENARIO_BLOCK_LENGTHS) < 2:
            raise ValueError("Specify at least two historical episode lengths.")
        if any(length < 12 for length in JOINT_SCENARIO_BLOCK_LENGTHS):
            raise ValueError("Every selection block length must be >= the forecast horizon.")
        block_length_records = []
        for episode_length in JOINT_SCENARIO_BLOCK_LENGTHS:
            episode_paths = calibrate_joint_historical_scenario_paths(
                historical_exog=ardl_training_data[ARDL_REGRESSORS],
                horizon=12,
                block_length=episode_length,
                adverse_directions=JOINT_SCENARIO_ADVERSE_DIRECTIONS,
                integrated_variables=JOINT_SCENARIO_INTEGRATED_VARIABLES,
                adverse_quantile=0.90,
                severe_quantile=0.975,
            )
            episode_forecasts = forecast_ardl_scenarios(
                fitted_results=joint_scenario_ardl_fit,
                scenarios=episode_paths,
                baseline_name="Baseline",
            )
            for scenario_name in ("Adverse", "Severe"):
                scenario_rows = episode_forecasts.loc[
                    episode_forecasts["scenario"] == scenario_name
                ].sort_values("period")
                metadata = episode_paths[scenario_name].attrs["calibration"]
                prefix = "severe" if scenario_name == "Severe" else "adverse"
                block_length_records.append({
                    "selection_block_length": episode_length,
                    "forecast_horizon": metadata["horizon"],
                    "candidate_blocks": metadata["candidate_blocks"],
                    "scenario": scenario_name,
                    "terminal_forecast": float(scenario_rows["forecast"].iloc[-1]),
                    "terminal_change_vs_baseline": float(
                        scenario_rows["change_vs_baseline"].iloc[-1]
                    ),
                    "selected_episode_start": metadata[f"{prefix}_block_start_period"],
                    "selected_episode_end": metadata[f"{prefix}_block_end_period"],
                    "forecast_shock_end": metadata[f"{prefix}_forecast_shock_end_period"],
                    "selected_episode_score": metadata[f"{prefix}_block_score"],
                })
        joint_block_length_sensitivity = pd.DataFrame(block_length_records)
        display(joint_block_length_sensitivity)
        joint_block_length_sensitivity.to_csv(
            PROCESSED / "ardl_joint_scenario_block_length_sensitivity.csv",
            index=False,
        )
        print("Saved joint scenario historical episode-length sensitivity results.")
else:
    print("Joint scenario calibration disabled. Review directions, integration labels and model evidence before enabling.")
